# Region Definition Verification
Visualizes the step-by-step construction of the L2/3 IT-rich region used for spike-in replacement.
Run this notebook from the `demo/data/` directory.

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import cv2
import matplotlib.pyplot as plt

HERE       = os.path.abspath('.')
RAW_DIR    = os.path.join(HERE, '../../../../ST/ALZ/alz-data/transcripts')
CELLS_FILE = os.path.join(HERE, '../../../../ST/ALZ/alz-data/SEAAD_MTG_MERFISH_metadata.2024-05-03.noblanks.harmonized.txt')
RESOLUTION_UM = 10

In [ ]:
# Discover all available samples
records = []
for donor in sorted(os.listdir(RAW_DIR)):
    donor_dir = os.path.join(RAW_DIR, donor)
    if not os.path.isdir(donor_dir):
        continue
    for id_ in sorted(os.listdir(donor_dir)):
        path = os.path.join(donor_dir, id_, 'cellpose-detected_transcripts.csv')
        if os.path.isfile(path):
            records.append({'sid': f'{donor}_{id_}', 'path': path})
all_samples = pd.DataFrame(records)
print(f'{len(all_samples)} samples found')
print(all_samples['sid'].tolist())

In [ ]:
# Pick a sample to inspect — change SID_TO_VIZ as desired
SID_TO_VIZ = all_samples['sid'].iloc[0]
raw_path   = all_samples.loc[all_samples.sid == SID_TO_VIZ, 'path'].iloc[0]
print(f'Visualizing: {SID_TO_VIZ}')

cells = pd.read_csv(CELLS_FILE, sep='\t')
cells['sid'] = cells.Section.str.split('_').str[0:2].str.join('_')

df = pd.read_csv(raw_path, dtype={9: str})
df = df[~df.gene.str.startswith('Blank')].reset_index(drop=True)

sid_cells = cells[cells.sid == SID_TO_VIZ]
l23it     = sid_cells[sid_cells.subclass_name == 'L2/3 IT']
other     = sid_cells[sid_cells.subclass_name != 'L2/3 IT']
print(f'{len(df):,} transcripts, {len(sid_cells):,} cells ({len(l23it):,} L2/3 IT)')

## Step 1: L2/3 IT cell locations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(other.x, other.y, s=1, c='lightgray', alpha=0.3, label='other')
ax.scatter(l23it.x, l23it.y, s=4, c='crimson',   alpha=0.8, label=f'L2/3 IT (n={len(l23it):,})')
ax.set_aspect('equal')
ax.legend(markerscale=4, fontsize=9)
ax.set_title(f'{SID_TO_VIZ} — cell types')
plt.tight_layout()
plt.show()

## Step 2: Rasterize L2/3 IT cells at 10 µm

In [ ]:
resolution = RESOLUTION_UM
x_min, x_max = df['x'].min(), df['x'].max()
y_min, y_max = df['y'].min(), df['y'].max()

xs = np.arange(x_min, x_max + resolution, resolution)
ys = np.arange(y_min, y_max + resolution, resolution)
layer = xr.DataArray(
    np.zeros((len(ys), len(xs)), dtype=np.uint8),
    dims=['y', 'x'],
    coords={'y': ys, 'x': xs},
)

for cx, cy in l23it[['x', 'y']].values:
    nearest = layer.sel(x=cx, y=cy, method='nearest')
    layer.loc[nearest.y.item(), nearest.x.item()] = 1

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(layer.values, origin='lower',
               extent=[xs[0], xs[-1], ys[0], ys[-1]], cmap='hot')
plt.colorbar(im, ax=ax, fraction=0.046, label='L2/3 IT cell present')
ax.set_aspect('equal')
ax.set_title(f'Rasterized L2/3 IT ({resolution} µm pixels)')
plt.tight_layout()
plt.show()
print(f'{layer.values.sum():,} occupied pixels out of {layer.values.size:,}')

## Step 3: Morphological close → open

In [ ]:
after_close = cv2.morphologyEx(
    layer.data, cv2.MORPH_CLOSE,
    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (40, 40))
)
after_open = cv2.morphologyEx(
    after_close, cv2.MORPH_OPEN,
    np.ones((10, 10), np.uint8)
)

titles = [
    'Thresholded (raw raster)',
    'After MORPH_CLOSE (40×40 ellipse)',
    'After MORPH_OPEN (10×10) — final region',
]
stages = [layer.data, after_close, after_open]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, data, title in zip(axes, stages, titles):
    ax.imshow(data, origin='lower',
              extent=[xs[0], xs[-1], ys[0], ys[-1]], cmap='binary')
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
plt.tight_layout()
plt.show()

region_mask = after_open.astype(bool)
print(f'Final region: {region_mask.sum():,} pixels = {region_mask.sum() * resolution**2 / 1e6:.2f} mm²')

## Step 4: Overlay region on transcripts

In [ ]:
# Map every transcript to its nearest grid pixel
xi_all = np.round((df['x'].values - xs[0]) / resolution).astype(int)
yi_all = np.round((df['y'].values - ys[0]) / resolution).astype(int)
xi_all = np.clip(xi_all, 0, len(xs) - 1)
yi_all = np.clip(yi_all, 0, len(ys) - 1)
in_region = region_mask[yi_all, xi_all]

# Subsample for plotting speed
rng_vis = np.random.default_rng(0)
idx_out = rng_vis.choice(np.where(~in_region)[0], size=min(40_000, (~in_region).sum()), replace=False)
idx_in  = rng_vis.choice(np.where(in_region)[0],  size=min(10_000, in_region.sum()),   replace=False)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df['x'].values[idx_out], df['y'].values[idx_out],
           s=0.2, c='lightgray', alpha=0.3, label='outside region')
ax.scatter(df['x'].values[idx_in], df['y'].values[idx_in],
           s=0.5, c='steelblue', alpha=0.6, label=f'in region ({in_region.sum():,} transcripts)')
ax.scatter(l23it.x, l23it.y, s=5, c='crimson', alpha=0.9, label='L2/3 IT cells')
ax.set_aspect('equal')
ax.set_xlabel('x (µm)')
ax.set_ylabel('y (µm)')
ax.legend(markerscale=5, fontsize=9)
ax.set_title(f'{SID_TO_VIZ}\nblue = transcripts replaced by SECRET; red = L2/3 IT cells')
plt.tight_layout()
plt.show()

print(f'Transcripts in region: {in_region.sum():,} / {len(df):,} ({100*in_region.mean():.1f}%)')
print(f'L2/3 IT cells in region: {region_mask[np.round((l23it.y.values - ys[0]) / resolution).astype(int).clip(0, len(ys)-1), np.round((l23it.x.values - xs[0]) / resolution).astype(int).clip(0, len(xs)-1)].sum()} / {len(l23it)}')